## Kieran Austin Quant Tennis Task
May 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')

from scripts.section_1 import reorder_participants_alphabetically, add_home_adv
from scripts.section_2 import set_validity_check, compare_surface, compare_round
from scripts.section_3 import filter_min_matches, model_a, model_b, check_adv_stats
from scripts.section_4 import model_c, run_match_simulations
from scripts.section_5 import rolling_forecast, evaluate_predictions, evaluation_summary, plot_calibration

In [ ]:
# import data 
df = pd.read_csv("../data/atp_results.csv")

## Section 1

In [ ]:
# 1.1 reorder participants alphabetically by surname
df_alpha = reorder_participants_alphabetically(df)

In [ ]:
# 1.2 add home_adv
df_alpha = add_home_adv(df_alpha)

## Section 2

In [ ]:
# 2.1check for instances of sum of sets not agreeing with who won
df_alpha = set_validity_check(df_alpha)

In [ ]:
# 2.1 Calculate games_per_set and clip a theorical upper bound
df_alpha['games_per_set'] = ((df_alpha['participant1_games_won'] + df_alpha['participant2_games_won']) / (df_alpha['participant1_sets_won'] + df_alpha['participant2_sets_won']))
df_alpha['games_per_set'] = df_alpha['games_per_set'].clip(upper=13)

In [ ]:
# 2.1.i compare surfaces
compare_surface(df_alpha)

In [ ]:
# 2.1.ii compare rounds
round_stats = compare_round(df_alpha)

In [ ]:
# 2.3 filter out matches with less than 10 matches
df_filtered = filter_min_matches(df_alpha, min_matches=10)

## Section 3

In [ ]:
# 3.2 model A
results_a = model_a(df_filtered)

In [ ]:
# 3.3 Model B
results_b = model_b(df_filtered)

In [ ]:
# check home advantage stats - as home is looking like a disadvantage
check_adv_stats(df_filtered, results_b)

## Section 4

In [ ]:
# 4.3 model c
results_c = model_c(df_filtered)

## Section 5

In [ ]:

# 5.1
# Model A
results_A = rolling_forecast(df_filtered, model='A', refit_freq='D')

# Model B
results_B = rolling_forecast(df_filtered, model='B', refit_freq='D', extra_columns=['home_adv'])

In [ ]:
# 5.1
# Model C - returns p_game; pass to Monte Carlo simulator next
results_C = rolling_forecast(df_filtered, model='C', refit_freq='W',
                              extra_columns=['best_of'])
# results_C['predicted_prob'] is per-game; rename for clarity:
results_C = results_C.rename(columns={'predicted_prob': 'p_game'})

# run monte carlo simulations
results_C = run_match_simulations(results_C, p_game_col='p_game',  best_of_col='best_of', output_col='predicted_prob_p1_wins', n_simulations=10000)

In [ ]:
# 5.2
# evaluate predictions
results_dict = {
    'Model A': (results_A, 'predicted_prob'),
    'Model B': (results_B, 'predicted_prob'),
    'Model C': (results_C, 'predicted_prob_p1_wins')}

# Metrics table
summary = evaluation_summary(results_dict)
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Calibration plot
fig = plot_calibration(results_dict, n_bins=10)
plt.savefig('../plots/calibration_models_abc.png', dpi=150, bbox_inches='tight')
plt.show()